Este cuaderno, como su nombre indica, es para realizar el entrenamiento y evaluación de modelos de vídeo.

Los modelos que vamos a trabajar en este cuaderno son los siguientes:
- *MCG-NJU/videomae-base* --> inspirado en cómo aprender los modelos de lenguaje (escondiendo palabras para que el modelo las adivine). Este modelo, toma un video, oculta el 90% de los píxeles (en forma de cubosde espacio-tiempo) y se fuerza a sí mismo a reconstruir lo que falta viendo solo el 10% restante.
- *facebook/timesformer-base-finetuned-k400* --> este modelo soluciona un problema crítico en este tipo de tareas: aplicar la atención (heredado de los transformers) sin tener que hacerlo a cada píxel (lo cual consume mucha memoria). Este modelo primero mira la relación espacial (píxeles dentro de un mismo frame) y luego la relación temporal (el mismo píxel a lo largo de los diferentes frames).
- *google/vivit-b-16x2-kinetics400* --> es la evolución directa de un ViT clásico, los cuales los hemos trabajado en el apartado anterior. Este modelo divide el vídeo en "Tubelets" o tubos 3D. Extrae un bloque de píxeles que atraviesa varios frames de golpe desde la primera capa.  

In [1]:
!pip install -q decord transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
import decord
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import evaluate
from transformers import (
    AutoImageProcessor,
    AutoModelForVideoClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    AutoProcessor
)
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
import random

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Entrenamiento Train/Valid

## 1.1. Selección del modelo y parámetros

In [4]:
# Descomentar el modelo que se quiera entrenar:
#MODELO_ELEGIDO = "videomae"
MODELO_ELEGIDO = "timesformer"
#MODELO_ELEGIDO = "vivit"

In [5]:
# --- PARAMETRIZACIÓN DE LA ETIQUETA ---

#ETIQUETA_OBJETIVO = "IDEOLOGICAL-INEQUALITY"
#ETIQUETA_OBJETIVO = "STEREOTYPING-DOMINANCE"
#ETIQUETA_OBJETIVO = "OBJECTIFICATION"
#ETIQUETA_OBJETIVO = "SEXUAL-VIOLENCE"
ETIQUETA_OBJETIVO = "MISOGYNY-NON-SEXUAL-VIOLENCE"

In [6]:
# Diccionario de Checkpoints (Hugging Face)
MODEL_ZOO = {
    "videomae": "MCG-NJU/videomae-base",
    "timesformer": "facebook/timesformer-base-finetuned-k400",
    "vivit": "google/vivit-b-16x2-kinetics400"
}

MODEL_CHECKPOINT = MODEL_ZOO[MODELO_ELEGIDO]

In [7]:
# Los modelos de vídeo suelen requerir un tamaño de clip fijo (ej. 16 o 8 frames)
# Para este experimento, extraemos 16 frames por vídeo para videome y timesformers o 8 frames para el vivit

if MODELO_ELEGIDO == "vivit":
  NUM_FRAMES_CLIP = 8
else:
  NUM_FRAMES_CLIP = 16

print(NUM_FRAMES_CLIP)

BATCH_SIZE = 4       # Este valor hay que mantenerlo bajo para evitar Out Of Memory en CUDA

16


In [8]:
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_train_3_3.csv"

# Output dinámico para no pisar modelos
OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Vision_Temporal/{ETIQUETA_OBJETIVO}/{MODELO_ELEGIDO.upper()}_FineTuned_experimentacion"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Configurado: {MODELO_ELEGIDO.upper()} con {NUM_FRAMES_CLIP} frames para la etiqueta {ETIQUETA_OBJETIVO}")

✅ Configurado: TIMESFORMER con 16 frames para la etiqueta MISOGYNY-NON-SEXUAL-VIOLENCE


In [9]:
decord.bridge.set_bridge('torch') # Silencia logs y une Decord con PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1.2. Carga y Partición de Datos

In [10]:
print(f"Lanzando experimento temporal con: {MODELO_ELEGIDO.upper()} para {ETIQUETA_OBJETIVO}")
df_train_master = pd.read_csv(CSV_TRAIN_MASTER)

# Adaptación a multietiqueta: renombramos la columna objetivo
df_train_master = df_train_master.rename(columns={ETIQUETA_OBJETIVO: "label"})
df_train_master["label"] = df_train_master["label"].astype(int)

# 90/10 Split estratificado
train_df, val_df = train_test_split(
    df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42
)

print("\nDistribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())
print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

Lanzando experimento temporal con: TIMESFORMER para MISOGYNY-NON-SEXUAL-VIOLENCE

Distribución fichero de entrenamiento (Train):
label
0    739
1    135
Name: count, dtype: int64

Distribución fichero de validación (Valid):
label
0    83
1    15
Name: count, dtype: int64


In [11]:
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        val_str = str(val)
        if not val_str.endswith(".mp4"): val_str += ".mp4"
        if "videos/" in val_str:
            ruta_completa = os.path.join(base_path, val_str)
        else:
            ruta_completa = os.path.join(base_path, "videos", val_str)
        rutas.append(ruta_completa)
    df['ruta_absoluta'] = rutas
    return df

train_df = fix_video_paths(train_df.copy(), RUTA_BASE_VIDEOS)
val_df = fix_video_paths(val_df.copy(), RUTA_BASE_VIDEOS)

## 1.3. Extracción de Frames y PyTorch dataset

In [12]:
print(f"Cargando procesador para {MODEL_CHECKPOINT}...")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT)
else:
  processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando procesador para facebook/timesformer-base-finetuned-k400...


preprocessor_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

In [13]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [14]:
# Entrenamiento SIN preprocesamiento

"""class VideoClassificationDataset(TorchDataset):
    def __init__(self, df, processor, clip_len=16):
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(self.clip_len, len(vr))
            frames = vr.get_batch(frame_indices).numpy()

            # Pasamos los frames de golpe
            if MODELO_ELEGIDO == "vivit":
                inputs = self.processor([list(frames)], return_tensors="pt")
            else:
                inputs = self.processor(list(frames), return_tensors="pt")

            pixel_values = inputs["pixel_values"][0]

            # Corrección exclusiva para ViViT
            if MODELO_ELEGIDO == "vivit":
              if pixel_values.shape[0] == 3:
                pixel_values = pixel_values.permute(1, 0, 2, 3)

        except Exception as e:
            # Salvavidas Seguro: Creamos un array numpy con ruido gris (128)
            # Esto evita los ceros puros que causan NaNs en las capas de normalización.

            if MODELO_ELEGIDO == "vivit":
                frames_falsos = np.ones((self.clip_len, 224, 224, 3), dtype=np.uint8) * 128
                inputs = self.processor([list(frames_falsos)], return_tensors="pt")
                pixel_values = inputs["pixel_values"][0]
            else:
                pixel_values = torch.zeros((self.clip_len, 3, 224, 224))

            etiqueta = 0 # Asumimos clase 0 para no sesgar

        return {"pixel_values": pixel_values, "label": etiqueta}

train_dataset = VideoClassificationDataset(train_df, processor, clip_len=NUM_FRAMES_CLIP)
valid_dataset = VideoClassificationDataset(val_df, processor, clip_len=NUM_FRAMES_CLIP)"""

# Entrenamiento CON preprocesamiento

#ESTRATEGIA_AUG = "50_percent_all"       # 50% de prob. a todos los vídeos (La más sana)
#ESTRATEGIA_AUG = "only_class_1"         # 100% de flip SOLO a etiqueta 1
#ESTRATEGIA_AUG = "only_class_0"         # 100% de flip SOLO a etiqueta 0
#ESTRATEGIA_AUG = "100_percent_all"      # 100% de flip a absolutamente todos
#ESTRATEGIA_AUG = "10_percent_class_1"   # 10% de prob. SOLO a etiqueta 1 (Para balanceo suave)
ESTRATEGIA_AUG = "none"                 # Sin preprocesamiento (Baseline puro)

class VideoClassificationDataset(TorchDataset):
    def __init__(self, df, processor, clip_len=16, is_training=False, aug_strategy="none"):
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len
        self.is_training = is_training
        self.aug_strategy = aug_strategy

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(self.clip_len, len(vr))
            frames = vr.get_batch(frame_indices).numpy()

            # ==========================================
            # 🔄 LÓGICA DE PREPROCESAMIENTO DINÁMICA
            # ==========================================
            if self.is_training:
                aplicar_flip = False

                if self.aug_strategy == "50_percent_all":
                    aplicar_flip = random.random() < 0.50
                elif self.aug_strategy == "only_class_1":
                    aplicar_flip = (etiqueta == 1)
                elif self.aug_strategy == "only_class_0":
                    aplicar_flip = (etiqueta == 0)
                elif self.aug_strategy == "100_percent_all":
                    aplicar_flip = True
                elif self.aug_strategy == "10_percent_class_1":
                    aplicar_flip = (etiqueta == 1 and random.random() < 0.10)

                # Si la lógica dictamina que sí, aplicamos el espejo a todos los frames
                if aplicar_flip:
                    frames = np.flip(frames, axis=2).copy()
            # ==========================================

            # Pasamos los frames de golpe
            if MODELO_ELEGIDO == "vivit":
                inputs = self.processor([list(frames)], return_tensors="pt")
            else:
                inputs = self.processor(list(frames), return_tensors="pt")

            pixel_values = inputs["pixel_values"][0]

            # Corrección exclusiva para ViViT
            if MODELO_ELEGIDO == "vivit":
              if pixel_values.shape[0] == 3:
                pixel_values = pixel_values.permute(1, 0, 2, 3)

        except Exception as e:
            # Salvavidas Seguro
            if MODELO_ELEGIDO == "vivit":
                frames_falsos = np.ones((self.clip_len, 224, 224, 3), dtype=np.uint8) * 128
                inputs = self.processor([list(frames_falsos)], return_tensors="pt")
                pixel_values = inputs["pixel_values"][0]
            else:
                pixel_values = torch.zeros((self.clip_len, 3, 224, 224))

            etiqueta = 0

        return {"pixel_values": pixel_values, "label": etiqueta}

# ---------------------------------------------------------
# INSTANCIACIÓN DE LOS DATASETS
# ---------------------------------------------------------
# Al train le pasamos la bandera is_training=True y la estrategia elegida arriba
train_dataset = VideoClassificationDataset(
    train_df, processor, clip_len=NUM_FRAMES_CLIP, is_training=True, aug_strategy=ESTRATEGIA_AUG
)

# Al valid SIEMPRE is_training=False y "none" para no alterar la evaluación
valid_dataset = VideoClassificationDataset(
    val_df, processor, clip_len=NUM_FRAMES_CLIP, is_training=False, aug_strategy="none"
)

In [15]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

## 1.4. Modelo y Métricas

In [16]:
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

In [17]:
model = AutoModelForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    #num_frames = NUM_FRAMES_CLIP, # Descomentar en casode entrenar modelo vitit
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/22.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  486MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

[transformers] TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [18]:
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

model.safetensors: reconstructing file:   0%|          |  0.00B /  486MB            

model.safetensors: downloading bytes:           |  0.00B            

In [19]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.5. Entrenamiento

In [20]:
# Hiperparámetros por defecto
learning_rate = 5e-5
num_train_epochs = 5
weight_decay = 0.01
warmup_ratio = 0.0
gradient_acumulation_steps = 2
lr_scheduler_type = 'linear'

# Slow cooker temporal
"""learning_rate=3e-5              # LR más bajito (bajamos de 5e-5 a 3e-5)
weight_decay=0.1                # Regularización altísima (subimos de 0.01 a 0.1)
num_train_epochs=8              # Le damos más tiempo para compensar la lentitud
warmup_ratio=0.1                # Calentamiento suave (10% del inicio) para no chocar
gradient_acumulation_steps = 2
lr_scheduler_type = 'linear'"""

# Gradientes Estables (Batch Size 16)
"""learning_rate = 5e-5
weight_decay = 0.01
num_train_epochs = 7
warmup_ratio = 0
gradient_acumulation_steps = 4
lr_scheduler_type = 'linear'"""

# LR Scheduler Agresivo (cosine)
"""learning_rate = 5e-5
weight_decay = 0.05             # Subimos un poco el weight_decay
num_train_epochs = 6
warmup_ratio = 0.1              # Fundamental para el coseno
gradient_acumulation_steps = 2
lr_scheduler_type = 'cosine'"""

tamaño_lote_efectivo = BATCH_SIZE * gradient_acumulation_steps
pasos_por_epoca = len(train_dataset) // tamaño_lote_efectivo
total_train_steps = pasos_por_epoca * num_train_epochs

warmup_steps_calculados = int(total_train_steps * warmup_ratio)

print(f"🔧 Lote efectivo: {tamaño_lote_efectivo}")
print(f"🔧 Pasos totales estimados: {total_train_steps}")
print(f"🔧 Pasos de calentamiento (warmup): {warmup_steps_calculados}")

🔧 Lote efectivo: 8
🔧 Pasos totales estimados: 545
🔧 Pasos de calentamiento (warmup): 0


In [21]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # CRUCIAL
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=gradient_acumulation_steps,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    warmup_steps=warmup_steps_calculados,
    #warmup_ratio=warmup_ratio, Deprecado
    lr_scheduler_type=lr_scheduler_type
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [23]:
print(f"\n🚀 Lanzando el entrenamiento temporal ({MODELO_ELEGIDO})...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))
print("✅ ¡Entrenamiento completado!")


🚀 Lanzando el entrenamiento temporal (timesformer)...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.876842,0.483290,0.455556,0.836735
2,0.540965,0.584549,0.482785,0.785714
3,0.200851,0.799282,0.581302,0.826531
4,0.033681,0.903438,0.610227,0.857143
5,0.007964,0.976048,0.610227,0.857143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Guardando el modelo definitivo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Resultados contra fichero de test estático

## 2.1. Configuración y rutas


In [24]:
# Rutas de Test Tarea 3.3
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/EXIST2025_test_3_3.csv"

# Guardamos en la carpeta dinámica que creamos arriba
CSV_SALIDA = f"{OUTPUT_DIR}/predicciones_{MODELO_ELEGIDO}_test_{ETIQUETA_OBJETIVO}.csv"
print(f"Las predicciones se guardarán en: {CSV_SALIDA}")

Las predicciones se guardarán en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Vision_Temporal/MISOGYNY-NON-SEXUAL-VIOLENCE/TIMESFORMER_FineTuned_experimentacion/predicciones_timesformer_test_MISOGYNY-NON-SEXUAL-VIOLENCE.csv


## 2.2. Carga de los datos (test) y del modelo

In [25]:
print("Cargando el dataset estático de test...")
test_df = pd.read_csv(CSV_TEST)

# Adaptación a multietiqueta
test_df = test_df.rename(columns={ETIQUETA_OBJETIVO: "label"})
test_df["label"] = test_df["label"].astype(int)

print("\nDistribución fichero de test:")
print(test_df['label'].value_counts())

Cargando el dataset estático de test...

Distribución fichero de test:
label
0    189
1     41
Name: count, dtype: int64


In [26]:
# Arreglar rutas absolutas (usamos la misma lógica que tenías)
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"): val = str(val) + ".mp4"
        rutas.append(os.path.join(base_path, val))
    df['ruta_absoluta'] = rutas
    return df

test_df = fix_video_paths(test_df, RUTA_BASE_VIDEOS)

In [27]:
# Función de muestreo de frames (se usa en el entrenamiento)
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [28]:
print(f"Cargando procesador y modelo {MODELO_ELEGIDO.upper()} entrenado...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")
else:
  processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")

model = AutoModelForVideoClassification.from_pretrained(OUTPUT_DIR + "/modelo_final").to(device)
model.eval()

print(f"Iniciando inferencia sobre {len(test_df)} vídeos de test...")

Cargando procesador y modelo TIMESFORMER entrenado...


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

Iniciando inferencia sobre 230 vídeos de test...


## 2.3. Inferencia directa video a video

In [29]:
y_true = []
y_pred = []
resultados_para_csv = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    ruta_video = row['ruta_absoluta']
    true_label = row['label']

    try:
        # Lectura eficiente con Decord
        vr = VideoReader(ruta_video, ctx=cpu(0))
        total_frames = len(vr)
        frame_indices = sample_frame_indices(NUM_FRAMES_CLIP, total_frames)

        # ⚠️ CORRECCIÓN AQUÍ: Usamos .numpy() en lugar de .asnumpy()
        frames = vr.get_batch(frame_indices).numpy()

        # Procesamos los 16 fotogramas de golpe
        if MODELO_ELEGIDO == "vivit":
          inputs = processor([list(frames)], return_tensors="pt")
        else:
          inputs = processor(list(frames), return_tensors="pt")

        pixel_values = inputs["pixel_values"].to(device)

        # Inferencia
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

    except Exception as e:
        print(f"\n⚠️ Error procesando {ruta_video}: {e}")
        # En caso de vídeo corrupto, asumimos incertidumbre (0.5) para no romper el test
        prob_misogino = 0.5

    # 1 si prob > 0.5, sino 0
    prediccion_binaria = 1 if prob_misogino > 0.5 else 0

    y_true.append(true_label)
    y_pred.append(prediccion_binaria)

    # Guardamos para el Ensemble Multimodal
    resultados_para_csv.append({
        "id_EXIST": id_vid,
        "prob_misogino_video": prob_misogino,
        "prediccion_binaria_video": prediccion_binaria,
        "label_real": true_label
    })

  1%|          | 2/230 [00:03<06:45,  1.78s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7305803156074597665.mp4: [21:04:32] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


100%|██████████| 230/230 [05:47<00:00,  1.51s/it]


## 2.4. Guardado del CSV de predicción y muestreo de métricas

In [30]:
# Guardamos el CSV
os.makedirs(os.path.dirname(CSV_SALIDA), exist_ok=True)
pd.DataFrame(resultados_para_csv).to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV de {MODELO_ELEGIDO} guardado para el Ensemble en: {CSV_SALIDA}!")


✅ ¡CSV de timesformer guardado para el Ensemble en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Task3.3/Research/Vision_Temporal/MISOGYNY-NON-SEXUAL-VIOLENCE/TIMESFORMER_FineTuned_experimentacion/predicciones_timesformer_test_MISOGYNY-NON-SEXUAL-VIOLENCE.csv!


In [31]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS TEST ESTÁTICO: {MODELO_ELEGIDO} (Análisis Temporal) con la etiqueta {ETIQUETA_OBJETIVO}")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS TEST ESTÁTICO: timesformer (Análisis Temporal) con la etiqueta MISOGYNY-NON-SEXUAL-VIOLENCE
F1-Score (Macro): 0.4615
Accuracy: 0.7913

Matriz de Confusión:
 [[181   8]
 [ 40   1]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.82      0.96      0.88       189
    Misógino       0.11      0.02      0.04        41

    accuracy                           0.79       230
   macro avg       0.47      0.49      0.46       230
weighted avg       0.69      0.79      0.73       230

